<a href="https://colab.research.google.com/github/lucasqueiros/hpc/blob/main/llm-lab/experiments/grupo_3/tokenization_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset e Tokenização

In [22]:
import tiktoken

In [23]:
enc = tiktoken.get_encoding("gpt2")

In [24]:
tokens = enc.encode("Hello, world!")
print(tokens)

[15496, 11, 995, 0]


In [25]:
enc.decode(tokens)

'Hello, world!'

In [26]:
tokensPortugues = enc.encode("Olá Brasil.")
print(tokensPortugues)

[30098, 6557, 39452, 346, 13]


In [27]:
enc.decode(tokensPortugues)

'Olá Brasil.'

In [28]:
print(len(tokens))
print(len(tokensPortugues))

4
5


**Implementando Dataset e DataLoader**

In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

In [30]:
from IPython.utils.path import target_outdated
class DataSet:
  def __init__(self,text ,tokenizer, context_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(text)

    for i in range(0, len(token_ids) - context_length, stride ):
      entrada = token_ids[i : i + context_length]
      target = token_ids[i + 1 : i + context_length + 1 ]

      self.input_ids.append(torch.tensor(entrada))
      self.target_ids.append(torch.tensor(target))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [31]:
def dataloader(text, batch_size=4, context_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = DataSet(text, tokenizer, context_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [32]:
import os
import requests

In [33]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [34]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text_verdict = f.read()

tokens_verdict = enc.encode(text_verdict)
print(f"The Verdict - Total de Tokens: {len(tokens_verdict)}")

The Verdict - Total de Tokens: 5145


In [35]:
if not os.path.exists("quixote.txt"):
    url = (
        "https://www.gutenberg.org/cache/epub/996/pg996.txt"
    )
    file_path = "quixote.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [36]:
with open("quixote.txt", "r", encoding="utf-8") as f:
    text_quixote = f.read()

tokens_quixote = enc.encode(text_quixote)
print(f"Dom Quixote - Total de Tokens: {len(tokens_quixote)}")

Dom Quixote - Total de Tokens: 621339


In [37]:
if not os.path.exists("shakespeare.txt"):
    url = (
        "https://www.gutenberg.org/files/100/100-0.txt"
    )
    file_path = "shakespeare.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [38]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text_shakespeare = f.read()

tokens_shakeaspeare = enc.encode(text_shakespeare)
print(f"Shakespeare - Total de Tokens: {len(tokens_shakeaspeare)}")

Shakespeare - Total de Tokens: 1686450


**Comparando resultados: Shakespeare e Don Quixote**

In [39]:
def calcular_metricas(texto, tokens):
  num_palavras = len(texto.split())
  num_tokens = len(tokens)
  return num_tokens / num_palavras

ratio_quixote = calcular_metricas(text_quixote, tokens_quixote)
ratio_shakespeare = calcular_metricas(text_shakespeare,tokens_shakeaspeare )

print(f"Tokens por palavra (Quixote): {ratio_quixote:.4f}")
print(f"Tokens por palavra (Shakespeare): {ratio_shakespeare:.4f}")

Tokens por palavra (Quixote): 1.4440
Tokens por palavra (Shakespeare): 1.7504


O livro dom quixote têm vocabulário mais específico ou construções que o tokenizador do GPT-2 precisa quebrar em mais subtokens, entretanto, as obras de Shakespeare necessitam muito mais da quebra em subtokens, visto que as palavras utilizadas são mais arcaicas.

In [40]:
if not os.path.exists("wiki.txt"):
    url = (
        "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/test.txt"
    )
    file_path = "wiki.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [41]:
with open("wiki.txt", "r", encoding="utf-8") as f:
    wiki_text = f.read()

tokens_wiki = enc.encode(wiki_text)

ratio_wiki = calcular_metricas(wiki_text, tokens_wiki)

print(f"Tokens por palavra (Wikipedia): {ratio_wiki:.4f}")
print(f"Total de Tokens: {len(tokens_wiki)}")

Tokens por palavra (Wikipedia): 1.2266
Total de Tokens: 295877


**Comparando resultados: Wikipédia, Shakespeare e Don Quixote**

In [42]:
print(f"Tokens por palavra (Quixote): {ratio_quixote:.4f}")
print(f"Tokens por palavra (Shakespeare): {ratio_shakespeare:.4f}")
print(f"Tokens por palavra (Wikipedia): {ratio_wiki:.4f}")
print("---------------------------------------------------")
print(f"Total de Tokens (Quixote): {len(tokens_quixote)}")
print(f"Total de Tokens (Shakespeare): {len(tokens_shakeaspeare)}")
print(f"Total de Tokens (Wikipedia): {len(tokens_wiki)}")



Tokens por palavra (Quixote): 1.4440
Tokens por palavra (Shakespeare): 1.7504
Tokens por palavra (Wikipedia): 1.2266
---------------------------------------------------
Total de Tokens (Quixote): 621339
Total de Tokens (Shakespeare): 1686450
Total de Tokens (Wikipedia): 295877


Comparação geral:
A variação nos índices de "Tokens por Palavra" prova que a eficiência de um LLM depende diretamente do quão próximo o texto de entrada está dos seus dados de treinamento. A Wikipédia é a mais eficiente por ser a base do conhecimento moderno do modelo, enquanto a obra de Shakespeare apresenta a maior fragmentação devido ao seu vocabulário mais específico e datado. Isso mostra que modelos de linguagem podem "sofrer" mais ao ler textos literários antigos do que artigos enciclopédicos modernos.